In [0]:

-- ============================================================================
-- PRICING FEATURES REVENUE ANALYSIS - CORRECTED
-- ============================================================================
-- Analyzes ONLY features where suppliers can set/control prices:
--
-- CORE PRICING MODELS:
-- 1. Individual Pricing (per person)
-- 2. Group Pricing (per group/private)
-- 3. Both Individual + Group (flexible pricing)
--
-- REVENUE ENHANCEMENT:
-- 4. Addons (upsells)
-- 5. Scale Pricing (volume discounts - GYG platform)
--
-- EXTERNAL/ADVANCED PRICING:
-- 6. Prices over API (supplier's own pricing system)
-- 7. Scales over API (supplier's own volume pricing system)
-- 8. Live Pricing & Availability (dynamic pricing capability)
-- ============================================================================

-- ----------------------------------------------------------------------------
-- BASE: Tour-level aggregation (no duplication)
-- ----------------------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW tour_pricing_base AS
SELECT 
  tour_id,
  supplier_id,
  segment,
  location_name,
  activity_type,
  
  -- CORE PRICING MODELS
  MAX(CASE WHEN has_individual_pricing = 1 THEN 1 ELSE 0 END) AS has_individual,
  MAX(CASE WHEN has_group_pricing = 1 THEN 1 ELSE 0 END) AS has_group,
  
  -- REVENUE ENHANCEMENT
  MAX(CASE WHEN has_addons_configured = 1 THEN 1 ELSE 0 END) AS has_addons,
  MAX(CASE WHEN has_any_scale_pricing = 1 THEN 1 ELSE 0 END) AS has_scale,
  
  -- EXTERNAL/ADVANCED
  MAX(CASE WHEN uses_prices_over_api = 1 THEN 1 ELSE 0 END) AS has_api_pricing,
  MAX(CASE WHEN uses_pricing_scales_over_api = 1 THEN 1 ELSE 0 END) AS has_api_scales,
  MAX(CASE WHEN uses_live_availability_and_price = 1 THEN 1 ELSE 0 END) AS has_live_pricing,
  
  -- Derived flags
  MAX(CASE WHEN uses_external_pricing = 1 THEN 1 ELSE 0 END) AS uses_external
  
FROM production.supply_analytics.pricing_feature_audit_base
GROUP BY tour_id, supplier_id, segment, location_name, activity_type;


CREATE OR REPLACE TEMP VIEW analysis_base AS
SELECT 
  t.*,
  
  -- PRICING MODEL CLASSIFICATION (mutually exclusive where possible)
  CASE 
    WHEN has_individual = 1 AND has_group = 1 THEN 'Individual + Group (Flexible)'
    WHEN has_individual = 1 AND has_group = 0 THEN 'Individual Only'
    WHEN has_individual = 0 AND has_group = 1 THEN 'Group Only'
    ELSE 'No Pricing Model'
  END AS pricing_model,
  
  -- FEATURE COMBINATIONS (key combos)
  CASE WHEN has_individual = 1 AND has_addons = 1 THEN 1 ELSE 0 END AS combo_ind_addons,
  CASE WHEN has_individual = 1 AND has_scale = 1 THEN 1 ELSE 0 END AS combo_ind_scale,
  CASE WHEN has_individual = 1 AND has_addons = 1 AND has_scale = 1 THEN 1 ELSE 0 END AS combo_ind_addons_scale,
  CASE WHEN has_individual = 1 AND has_group = 1 AND has_addons = 1 THEN 1 ELSE 0 END AS combo_flex_addons,
  CASE WHEN has_group = 1 AND has_addons = 1 THEN 1 ELSE 0 END AS combo_group_addons,
  
  -- NATIVE vs EXTERNAL
  CASE 
    WHEN has_api_pricing = 1 OR has_api_scales = 1 THEN 'External API'
    WHEN has_live_pricing = 1 THEN 'Live Dynamic'
    ELSE 'Native GYG'
  END AS pricing_system,
  
  -- Feature count (pricing features only)
  (has_individual + has_group + has_addons + has_scale) AS pricing_feature_count,
  
  -- Performance
  p.gmv_l12mo,
  p.bookings_l12mo,
  p.avg_booking_value_l12mo,
  p.repeat_customer_rate_l12mo,
  p.customers_l12mo
  
FROM tour_pricing_base t
INNER JOIN production.supply_analytics.pricing_feature_audit_performance p
  ON t.tour_id = p.tour_id
WHERE p.gmv_l12mo > 0;


-- ----------------------------------------------------------------------------
-- ANALYSIS 1: PRICING MODEL PERFORMANCE (Individual vs Group vs Both)
-- ----------------------------------------------------------------------------

CREATE OR REPLACE TABLE production.supply_analytics.pricing_model_performance AS

SELECT 
  pricing_model,
  COUNT(DISTINCT tour_id) AS tours,
  COUNT(DISTINCT supplier_id) AS suppliers,
  
  ROUND(AVG(gmv_l12mo), 0) AS avg_gmv_per_tour,
  ROUND(PERCENTILE(gmv_l12mo, 0.5), 0) AS median_gmv_per_tour,
  ROUND(SUM(gmv_l12mo) / 1000000, 1) AS total_gmv_millions,
  
  ROUND(AVG(avg_booking_value_l12mo), 2) AS avg_aov,
  ROUND(AVG(bookings_l12mo), 1) AS avg_bookings_per_tour,
  ROUND(AVG(customers_l12mo), 1) AS avg_customers_per_tour,
  ROUND(AVG(repeat_customer_rate_l12mo), 1) AS avg_repeat_rate_pct
  
FROM analysis_base
GROUP BY pricing_model
ORDER BY avg_gmv_per_tour DESC;


-- ----------------------------------------------------------------------------
-- ANALYSIS 2: FEATURE COMBINATIONS (Native Pricing Only)
-- ----------------------------------------------------------------------------

CREATE OR REPLACE TABLE production.supply_analytics.pricing_feature_combinations AS

SELECT 
  feature_combo,
  native_only,
  tours,
  suppliers,
  avg_gmv,
  median_gmv,
  total_gmv_millions,
  avg_aov,
  
  -- Calculate vs baseline
  ROUND(100.0 * (avg_gmv - baseline_gmv) / NULLIF(baseline_gmv, 0), 1) AS gmv_lift_vs_baseline_pct
  
FROM (
  SELECT 
    'Individual Only' AS feature_combo,
    1 AS native_only,
    COUNT(DISTINCT tour_id) AS tours,
    COUNT(DISTINCT supplier_id) AS suppliers,
    ROUND(AVG(gmv_l12mo), 0) AS avg_gmv,
    ROUND(PERCENTILE(gmv_l12mo, 0.5), 0) AS median_gmv,
    ROUND(SUM(gmv_l12mo) / 1000000, 1) AS total_gmv_millions,
    ROUND(AVG(avg_booking_value_l12mo), 2) AS avg_aov
  FROM analysis_base
  WHERE pricing_model = 'Individual Only' 
    AND has_addons = 0 AND has_scale = 0 
    AND uses_external = 0
  
  UNION ALL
  
  SELECT 'Group Only', 1,
    COUNT(DISTINCT tour_id), COUNT(DISTINCT supplier_id),
    ROUND(AVG(gmv_l12mo), 0), ROUND(PERCENTILE(gmv_l12mo, 0.5), 0),
    ROUND(SUM(gmv_l12mo) / 1000000, 1), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base
  WHERE pricing_model = 'Group Only' 
    AND has_addons = 0 AND has_scale = 0 
    AND uses_external = 0
  
  UNION ALL
  
  SELECT 'Individual + Group (Flexible)', 1,
    COUNT(DISTINCT tour_id), COUNT(DISTINCT supplier_id),
    ROUND(AVG(gmv_l12mo), 0), ROUND(PERCENTILE(gmv_l12mo, 0.5), 0),
    ROUND(SUM(gmv_l12mo) / 1000000, 1), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base
  WHERE pricing_model = 'Individual + Group (Flexible)' 
    AND has_addons = 0 AND has_scale = 0 
    AND uses_external = 0
  
  UNION ALL
  
  SELECT 'Individual + Addons', 1,
    COUNT(DISTINCT tour_id), COUNT(DISTINCT supplier_id),
    ROUND(AVG(gmv_l12mo), 0), ROUND(PERCENTILE(gmv_l12mo, 0.5), 0),
    ROUND(SUM(gmv_l12mo) / 1000000, 1), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base
  WHERE combo_ind_addons = 1 AND has_scale = 0 AND has_group = 0 AND uses_external = 0
  
  UNION ALL
  
  SELECT 'Individual + Scale', 1,
    COUNT(DISTINCT tour_id), COUNT(DISTINCT supplier_id),
    ROUND(AVG(gmv_l12mo), 0), ROUND(PERCENTILE(gmv_l12mo, 0.5), 0),
    ROUND(SUM(gmv_l12mo) / 1000000, 1), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base
  WHERE combo_ind_scale = 1 AND has_addons = 0 AND has_group = 0 AND uses_external = 0
  
  UNION ALL
  
  SELECT 'Individual + Addons + Scale', 1,
    COUNT(DISTINCT tour_id), COUNT(DISTINCT supplier_id),
    ROUND(AVG(gmv_l12mo), 0), ROUND(PERCENTILE(gmv_l12mo, 0.5), 0),
    ROUND(SUM(gmv_l12mo) / 1000000, 1), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base
  WHERE combo_ind_addons_scale = 1 AND has_group = 0 AND uses_external = 0
  
  UNION ALL
  
  SELECT 'Flexible + Addons', 1,
    COUNT(DISTINCT tour_id), COUNT(DISTINCT supplier_id),
    ROUND(AVG(gmv_l12mo), 0), ROUND(PERCENTILE(gmv_l12mo, 0.5), 0),
    ROUND(SUM(gmv_l12mo) / 1000000, 1), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base
  WHERE combo_flex_addons = 1 AND has_scale = 0 AND uses_external = 0
  
  UNION ALL
  
  SELECT 'Group + Addons', 1,
    COUNT(DISTINCT tour_id), COUNT(DISTINCT supplier_id),
    ROUND(AVG(gmv_l12mo), 0), ROUND(PERCENTILE(gmv_l12mo, 0.5), 0),
    ROUND(SUM(gmv_l12mo) / 1000000, 1), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base
  WHERE combo_group_addons = 1 AND has_scale = 0 AND has_individual = 0 AND uses_external = 0
  
  UNION ALL
  
  SELECT 'Full Featured (Flex + Addons + Scale)', 1,
    COUNT(DISTINCT tour_id), COUNT(DISTINCT supplier_id),
    ROUND(AVG(gmv_l12mo), 0), ROUND(PERCENTILE(gmv_l12mo, 0.5), 0),
    ROUND(SUM(gmv_l12mo) / 1000000, 1), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base
  WHERE has_individual = 1 AND has_group = 1 AND has_addons = 1 AND has_scale = 1 AND uses_external = 0
) combos
CROSS JOIN (
  SELECT ROUND(AVG(gmv_l12mo), 0) AS baseline_gmv
  FROM analysis_base 
  WHERE pricing_model = 'Individual Only' AND has_addons = 0 AND has_scale = 0 AND uses_external = 0
) baseline
ORDER BY avg_gmv DESC;


-- ----------------------------------------------------------------------------
-- ANALYSIS 3: EXTERNAL vs NATIVE PRICING SYSTEMS
-- ----------------------------------------------------------------------------

CREATE OR REPLACE TABLE production.supply_analytics.pricing_system_performance AS

SELECT 
  pricing_system,
  COUNT(DISTINCT tour_id) AS tours,
  COUNT(DISTINCT supplier_id) AS suppliers,
  
  ROUND(AVG(gmv_l12mo), 0) AS avg_gmv,
  ROUND(PERCENTILE(gmv_l12mo, 0.5), 0) AS median_gmv,
  ROUND(SUM(gmv_l12mo) / 1000000, 1) AS total_gmv_millions,
  
  ROUND(AVG(avg_booking_value_l12mo), 2) AS avg_aov,
  ROUND(AVG(bookings_l12mo), 1) AS avg_bookings
  
FROM analysis_base
GROUP BY pricing_system
ORDER BY avg_gmv DESC;


-- ----------------------------------------------------------------------------
-- ANALYSIS 4: INDIVIDUAL FEATURES BY SEGMENT (Native Only)
-- ----------------------------------------------------------------------------

CREATE OR REPLACE TABLE production.supply_analytics.pricing_features_by_segment AS

WITH segment_metrics AS (
  SELECT 
    segment,
    'Individual Pricing' AS feature_name,
    has_individual AS has_feature,
    COUNT(DISTINCT tour_id) AS tours,
    ROUND(AVG(gmv_l12mo), 0) AS avg_gmv,
    ROUND(AVG(avg_booking_value_l12mo), 2) AS avg_aov
  FROM analysis_base WHERE uses_external = 0 GROUP BY segment, has_individual
  
  UNION ALL
  
  SELECT segment, 'Group Pricing', has_group,
    COUNT(DISTINCT tour_id), ROUND(AVG(gmv_l12mo), 0), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base WHERE uses_external = 0 GROUP BY segment, has_group
  
  UNION ALL
  
  SELECT segment, 'Addons', has_addons,
    COUNT(DISTINCT tour_id), ROUND(AVG(gmv_l12mo), 0), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base WHERE uses_external = 0 GROUP BY segment, has_addons
  
  UNION ALL
  
  SELECT segment, 'Scale Pricing', has_scale,
    COUNT(DISTINCT tour_id), ROUND(AVG(gmv_l12mo), 0), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base WHERE uses_external = 0 GROUP BY segment, has_scale
  
  UNION ALL
  
  SELECT segment, 'API Pricing', has_api_pricing,
    COUNT(DISTINCT tour_id), ROUND(AVG(gmv_l12mo), 0), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base GROUP BY segment, has_api_pricing
  
  UNION ALL
  
  SELECT segment, 'Live Pricing', has_live_pricing,
    COUNT(DISTINCT tour_id), ROUND(AVG(gmv_l12mo), 0), ROUND(AVG(avg_booking_value_l12mo), 2)
  FROM analysis_base GROUP BY segment, has_live_pricing
)

SELECT 
  segment,
  feature_name,
  
  MAX(CASE WHEN has_feature = 1 THEN tours END) AS tours_with,
  MAX(CASE WHEN has_feature = 0 THEN tours END) AS tours_without,
  
  ROUND(100.0 * MAX(CASE WHEN has_feature = 1 THEN tours END) / 
    NULLIF(SUM(tours), 0), 1) AS adoption_pct,
  
  MAX(CASE WHEN has_feature = 1 THEN avg_gmv END) AS avg_gmv_with,
  MAX(CASE WHEN has_feature = 0 THEN avg_gmv END) AS avg_gmv_without,
  
  ROUND(100.0 * (MAX(CASE WHEN has_feature = 1 THEN avg_gmv END) - 
                 MAX(CASE WHEN has_feature = 0 THEN avg_gmv END)) /
    NULLIF(MAX(CASE WHEN has_feature = 0 THEN avg_gmv END), 0), 1) AS gmv_lift_pct

FROM segment_metrics
GROUP BY segment, feature_name
HAVING tours_with IS NOT NULL AND tours_without IS NOT NULL
ORDER BY segment, gmv_lift_pct DESC;


-- ----------------------------------------------------------------------------
-- ANALYSIS 5: SEGMENT SUMMARY
-- ----------------------------------------------------------------------------

CREATE OR REPLACE TABLE production.supply_analytics.segment_pricing_summary AS

SELECT 
  segment,
  COUNT(DISTINCT tour_id) AS tours,
  COUNT(DISTINCT supplier_id) AS suppliers,
  
  ROUND(AVG(pricing_feature_count), 1) AS avg_pricing_features,
  ROUND(AVG(gmv_l12mo), 0) AS avg_gmv,
  ROUND(AVG(avg_booking_value_l12mo), 2) AS avg_aov,
  
  -- Adoption rates
  ROUND(100.0 * SUM(CASE WHEN has_individual = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS individual_adoption_pct,
  ROUND(100.0 * SUM(CASE WHEN has_group = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS group_adoption_pct,
  ROUND(100.0 * SUM(CASE WHEN has_addons = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS addons_adoption_pct,
  ROUND(100.0 * SUM(CASE WHEN has_scale = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS scale_adoption_pct,
  ROUND(100.0 * SUM(CASE WHEN has_api_pricing = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS api_pricing_adoption_pct,
  ROUND(100.0 * SUM(CASE WHEN has_live_pricing = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS live_pricing_adoption_pct,
  
  -- Pricing model distribution
  ROUND(100.0 * SUM(CASE WHEN pricing_model = 'Individual Only' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_individual_only,
  ROUND(100.0 * SUM(CASE WHEN pricing_model = 'Group Only' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_group_only,
  ROUND(100.0 * SUM(CASE WHEN pricing_model = 'Individual + Group (Flexible)' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_flexible

FROM analysis_base
GROUP BY segment
ORDER BY avg_gmv DESC;


-- View results
SELECT * FROM production.supply_analytics.pricing_model_performance;
SELECT * FROM production.supply_analytics.pricing_feature_combinations;
SELECT * FROM production.supply_analytics.pricing_system_performance;
SELECT * FROM production.supply_analytics.pricing_features_by_segment;
SELECT * FROM production.supply_analytics.segment_pricing_summary;
